In [2]:
import polars as pl
import time

RAW_DATA_PATH = "../data/raw/2019-Oct.csv"
PROCESSED_DATA_PATH = "../data/processed/ecommerce_cleaned.parquet"

pl.Config.set_fmt_float("full")
print("Library loaded successfully")

Library loaded successfully


In [8]:
print("Checking")
start_time = time.time()
df_lazy = pl.scan_csv(RAW_DATA_PATH)
print(f"Done checking:{round(time.time()-start_time, 4)} seconds")
df_lazy.head(5).collect()

Checking
Done checking:0.0271 seconds


event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,i64,i64,str,str,f64,i64,str
"""2019-10-01 00:00:00 UTC""","""view""",44600062,2103807459595387724,null,"""shiseido""",35.79,541312140,"""72d76fde-8bb3-4e00-8c23-a032df…"
"""2019-10-01 00:00:00 UTC""","""view""",3900821,2053013552326770905,"""appliances.environment.water_h…","""aqua""",33.2,554748717,"""9333dfbd-b87a-4708-9857-633655…"
"""2019-10-01 00:00:01 UTC""","""view""",17200506,2053013559792632471,"""furniture.living_room.sofa""",null,543.1,519107250,"""566511c2-e2e3-422b-b695-cf8e6e…"
"""2019-10-01 00:00:01 UTC""","""view""",1307067,2053013558920217191,"""computers.notebook""","""lenovo""",251.74,550050854,"""7c90fc70-0e80-4590-96f3-13c02c…"
"""2019-10-01 00:00:04 UTC""","""view""",1004237,2053013555631882655,"""electronics.smartphone""","""apple""",1081.98,535871217,"""c6bd7419-2748-4c56-95b4-8cec9f…"


In [ ]:
print("Transformation start")
transform_start = time.time()
df_cleaned = (df_lazy
              .drop_nulls(subset=["product_id", "category_id", "user_id"])
              .with_columns(pl.col("event_time").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S %Z"))
              .with_columns([pl.col("category_code").str.split(".").list.get(0).alias("main_category")])
              .filter(pl.col("price") > 0)
              .collect()
              )
print(f"Processing...:{round(time.time()- transform_start, 2)} seconds")
print(f"Cleaned Data:{df_cleaned.height}")

df_cleaned.head(5)

Transformation start
Process...:11.81 seconds
Cleaned Data:42380091


event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,main_category
datetime[μs],str,i64,i64,str,str,f64,i64,str,str
2019-10-01 00:00:00,"""view""",44600062,2103807459595387724,null,"""shiseido""",35.79,541312140,"""72d76fde-8bb3-4e00-8c23-a032df…",null
2019-10-01 00:00:00,"""view""",3900821,2053013552326770905,"""appliances.environment.water_h…","""aqua""",33.2,554748717,"""9333dfbd-b87a-4708-9857-633655…","""appliances"""
2019-10-01 00:00:01,"""view""",17200506,2053013559792632471,"""furniture.living_room.sofa""",null,543.1,519107250,"""566511c2-e2e3-422b-b695-cf8e6e…","""furniture"""
2019-10-01 00:00:01,"""view""",1307067,2053013558920217191,"""computers.notebook""","""lenovo""",251.74,550050854,"""7c90fc70-0e80-4590-96f3-13c02c…","""computers"""
2019-10-01 00:00:04,"""view""",1004237,2053013555631882655,"""electronics.smartphone""","""apple""",1081.98,535871217,"""c6bd7419-2748-4c56-95b4-8cec9f…","""electronics"""


In [7]:
print("Extracting to Parquet...")
write_start = time.time()
df_cleaned.write_parquet(PROCESSED_DATA_PATH)
print(f"Processing...: {round(time.time() - write_start, 2)} seconds")
print("Done pipeline, ready for BigQuery")

Extracting to Parquet...
Processing...: 4.11 seconds
Done pipeline, ready for BigQuery
